<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Data Preprocessing and Feature Engineering

*Session 5 · Notebook 01.03 · Lecture · Student version (v2 draft)*

## Overview

Before any model can learn, the data has to be prepared. This notebook covers the core preprocessing steps in scikit-learn: handling missing values (numeric and categorical), scaling (normalization and standardization), engineering new features, encoding categorical variables, dealing with class imbalance, and splitting the data so that preprocessing does not leak information from the test set. It finishes with the professional way to wire all of this together, a scikit-learn **Pipeline**.

The running example is the **German Credit** dataset (a risk dataset), with a small synthetic table for the missing-data and date sections. The exercises use the seaborn **tips** dataset.

## Learning Objectives

By the end of this notebook you will be able to:

- Impute missing values for both numeric and categorical columns with `SimpleImputer`.
- Scale features with `MinMaxScaler` and `StandardScaler`, and see the impact visually.
- Engineer new features (date parts, bins, ratios and interactions).
- Encode categorical variables with label and one-hot encoding.
- Rebalance an imbalanced target with over- and under-sampling.
- Split data and preprocess without leakage, and assemble a `Pipeline`.

## Prerequisites

- Session 4 (distributions, skew, outliers, missing-data mechanisms).
- Session 2/3 pandas (`read_csv`, `isna`, selecting columns).

## Index

1. [Why this matters for risk modelling](#sec1)
2. [Handling missing data (numeric and categorical)](#sec2)
3. [Normalization (Min-Max scaling)](#sec3)
4. [Standardization (Z-scaling)](#sec4)
5. [Feature engineering](#sec5)
6. [Encoding categorical variables](#sec6)
7. [Handling imbalanced data](#sec7)
8. [Splitting and preprocessing without leakage](#sec8)
9. [Putting it together: a preprocessing Pipeline](#sec9)
10. [Exercises](#exercises)
11. [Challenge](#challenge)
12. [Key Takeaways](#takeaways)
13. [Further Reading](#reading)

<a id="setup"></a>
# Section 0: Setup

The running example dataset is `german_credit.csv` (1,000 loan applicants with an imbalanced good/bad target), read from the repo-root `datasets/` folder (two levels up). The exercises use the seaborn `tips` dataset. A small synthetic table is built in Section 2.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sns.set_theme(style='whitegrid')
np.random.seed(42)

# Or read directly from the public S3 bucket (no local file needed):
# credit = pd.read_csv('https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/Data_Sources_CBS_Risk/Session_5/german_credit.csv')   # running example
# ...or read the paths from a config file (the local read below stays the default):
# from config import session_datasets_http
# credit = pd.read_csv(session_datasets_http["german_credit"])   # running example
# Or from S3 with Spark, then to pandas (needs a SparkSession, e.g. on Databricks):
# credit = spark.read.csv("s3://rockborne-bucket-01-cbs/Data_Sources_CBS_Risk/Session_5/german_credit.csv", header=True, inferSchema=True).toPandas()   # running example
credit = pd.read_csv('../../datasets/Session_5/german_credit.csv')   # running example
# Seaborn source (kept for reuse with other clients); to use it, swap the read_csv line for:
# tips = sns.load_dataset('tips')                             # used in the exercises
# Or read directly from the public S3 bucket (no local file needed):
# tips = pd.read_csv('https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/Data_Sources_CBS_Risk/Session_5/tips.csv')                             # used in the exercises
# ...or read the paths from a config file (the local read below stays the default):
# from config import session_datasets_http
# tips = pd.read_csv(session_datasets_http["tips"])                             # used in the exercises
# Or from S3 with Spark, then to pandas (needs a SparkSession, e.g. on Databricks):
# tips = spark.read.csv("s3://rockborne-bucket-01-cbs/Data_Sources_CBS_Risk/Session_5/tips.csv", header=True, inferSchema=True).toPandas()                             # used in the exercises
tips = pd.read_csv('../../datasets/Session_5/tips.csv')                             # used in the exercises
print('german_credit:', credit.shape, '| tips:', tips.shape)
credit.head()

<a id="sec1"></a>
# Section 1: Why this matters for risk modelling

A credit or fraud model is only as good as the data fed into it, and raw risk data is rarely model-ready.

| Preprocessing step | Why a risk model needs it |
|---|---|
| **Missing values** | Application data has gaps, in text fields as well as numbers; most models cannot train with `NaN`. |
| **Scaling** | Distance- and gradient-based models (KNN, SVM, regularised regression) are distorted when amount is in thousands and age in tens. |
| **Feature engineering** | A ratio like instalment-to-income often predicts default better than either raw column. |
| **Encoding** | Models need numbers, but categories like loan purpose have no natural order. |
| **Class imbalance** | Defaults and fraud are rare; without rebalancing a model can score everyone 'good' and still look accurate. |
| **No leakage** | If scaling or resampling sees the test set, validation results are optimistic and the model fails in production. |

Getting preprocessing wrong does not throw an error; it silently produces a model that looks fine in testing and misbehaves in production.

<a id="sec2"></a>
# Section 2: Handling missing data (numeric and categorical)

**Definition:** imputation replaces missing values with estimates so the data is complete enough to model (the statistical side, and the MCAR/MAR/MNAR mechanisms, were covered in Session 4).

**Example:** an application form left the income field blank (numeric) and the region field blank (categorical); we fill each appropriately.

**Analogy:** filling gaps in a form with the typical answer so the form can be processed.

**Explanation:** `SimpleImputer` fills a column with a statistic. For **numeric** columns use the **mean** (or median for skewed money columns); for **categorical** columns you cannot take a mean, so use the **most frequent** value. We use a small synthetic customer table that has gaps in both kinds of column (and a date column we reuse in Section 5).

In [ ]:
# A small synthetic dataset with missing values in numeric AND categorical columns
# A small synthetic dataset with missing values in numeric AND categorical columns
synth = pd.DataFrame({
    'region':      ['North', 'South', np.nan, 'East', 'West', 'North', np.nan, 'South', 'East', 'West'],
    'age':         [34, 45, 29, np.nan, 52, 41, 38, np.nan, 27, 60],
    'income':      [42000, 58000, np.nan, 61000, 73000, np.nan, 52000, 48000, 39000, 85000],
    'signup_date': pd.to_datetime(['2021-01-15', '2021-03-22', '2021-07-01', '2022-02-10',
                                   '2022-05-30', '2022-08-19', '2023-01-05', '2023-04-12',
                                   '2023-09-27', '2023-12-03']),
    'subscribed':  ['Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'No', 'Yes'],
})
print('Missing values per column:')
print(synth.isna().sum())
synth

### Impute numeric columns with the mean

In [ ]:
# Your turn. Write your solution here:

### Impute categorical columns with the most frequent value

A category has no mean, so we fill blanks with the **mode** (`strategy='most_frequent'`).

In [ ]:
# Your turn. Write your solution here:

<a id="sec3"></a>
# Section 3: Normalization (Min-Max scaling)

**Definition:** normalization rescales each feature to a fixed range, usually 0 to 1.

**Example:** putting `credit_amount` (thousands), `duration_months` (tens) and `age_years` (tens) onto the same 0-to-1 scale.

**Analogy:** redrawing several charts so they all use the same axis, making them comparable.

**Explanation:** `MinMaxScaler` maps the smallest value to 0 and the largest to 1. It keeps the shape of the distribution and is useful when a model expects bounded inputs, but it is sensitive to outliers. The plot below shows the impact: before scaling, `credit_amount` dwarfs the other features; after, all three are comparable.

<a id="sec4"></a>
# Section 4: Standardization (Z-scaling)

**Definition:** standardization rescales each feature to mean 0 and standard deviation 1 (a z-score).

**Example:** expressing each credit amount as 'how many standard deviations from the average'.

**Analogy:** grading on a curve: every subject is expressed relative to its own average and spread, so they can be compared fairly.

**Explanation:** `StandardScaler` is the default for most models (regularised regression, SVM, PCA) because it centres the data and equalises variance. Use standardization by default; reach for normalization when a method needs a fixed input range. The plot shows every feature now centred on 0 with comparable spread.

### Try it yourself

Standardize the credit `installment_rate` and `existing_credits` columns with `StandardScaler`, and confirm each has mean about 0.

<a id="sec5"></a>
# Section 5: Feature engineering

**Definition:** feature engineering creates new, more informative columns from existing ones.

**Example:** turning a signup date into a customer tenure, or `credit_amount` and `duration_months` into a monthly-instalment estimate.

**Analogy:** a chef combining raw ingredients into a dish: the combination is worth more than the parts.

**Explanation:** good features often matter more than the choice of model. Three common techniques: extracting parts of a date, binning a continuous variable, and building ratios or interactions.

### 5.1 Extracting parts of a date

Using the synthetic table's `signup_date`, we pull out calendar parts and derive a tenure (days since signup), which is often more predictive than the raw date.

### 5.2 Binning a continuous variable

`pd.cut` groups a numeric column into labelled bands, which can capture non-linear effects (risk that differs by age band rather than smoothly with age).

### 5.3 Ratios and interactions

Combining columns often creates the most predictive feature. Here, an estimated monthly instalment (amount divided by duration) captures repayment burden.

### Try it yourself

Create a feature `amount_per_year_of_age` = `credit_amount` divided by `age_years` and print its first few values.

<a id="sec6"></a>
# Section 6: Encoding categorical variables

**Definition:** encoding converts categorical text into numbers a model can use.

**Example:** turning loan `purpose` (car, education, business, ...) into numeric columns.

**Analogy:** translating words into a language (numbers) the model speaks.

**Explanation:** two main approaches:

- **Label encoding** maps each category to an integer. Compact, but it **invents an order**, which misleads models for unordered categories. Use it only for ordinal data or tree models.
- **One-hot encoding** creates a 0/1 column per category, with no false ordering. The safe default for nominal categories, at the cost of more columns.

In [ ]:
# Label encoding: one integer per category (note the invented order)
# One-hot encoding: a 0/1 column per category (no false ordering)

### Try it yourself

One-hot encode `checking_status` with `pd.get_dummies` (prefix `chk`). How many columns does it produce?

In [ ]:
# Your turn. Write your solution here:


<a id="sec7"></a>
# Section 7: Handling imbalanced data

**Definition:** a target is imbalanced when one class is much rarer than the other.

**Example:** in the credit data, 30% are bad risks; in fraud, the rare class can be under 1%.

**Analogy:** studying for an exam where 99 questions are on one topic and 1 on another: you would barely learn the rare topic. A model trained on imbalanced data behaves the same way.

**Explanation:** with heavy imbalance a model can score everything as the majority class and still look accurate while missing every rare case. **Oversampling** duplicates or synthesises minority examples; **undersampling** drops majority examples. `imblearn` provides both (and SMOTE). **Crucial:** rebalance the *training* data only, never the test set. Below we oversample the credit risk classes and visualise the effect.

<a id="sec8"></a>
# Section 8: Splitting and preprocessing without leakage

**Definition:** data leakage is when information from the test set influences preprocessing, making validation results too optimistic.

**Example:** fitting a `StandardScaler` on the whole dataset means the test set's mean and range have secretly shaped the scaling.

**Analogy:** letting students see the exam answers while revising: their practice scores look great but mean nothing.

**Explanation:** the rule is simple. **Split first**, then **fit** every transformer on the **training set only**, and merely **apply** them to the test set.

<a id="sec9"></a>
# Section 9: Putting it together - a preprocessing Pipeline

**Definition:** a scikit-learn **Pipeline** chains preprocessing steps (and a model) into one object; a **ColumnTransformer** applies different steps to different columns.

**Example:** scale the numeric columns and one-hot encode the categorical columns in a single fitted object.

**Analogy:** a factory production line: raw data goes in one end and model-ready data comes out the other, identically every time.

**Explanation:** a Pipeline is the professional way to preprocess: it **guarantees no leakage** (everything is fitted on train when you call `fit`), keeps the steps together so they apply identically in production, and can be cross-validated as a unit.

<a id="exercises"></a>
# Section 10: Exercises

These use the seaborn `tips` dataset (restaurant bills), a change from the credit data used in the examples.

### Exercise 1: Normalize vs standardize

Take `tips[['total_bill', 'tip', 'size']]`. Produce a Min-Max normalized version and a standardized version, and print the min/max of the normalized one and the mean of the standardized one.

In [ ]:
# Your turn. Write your solution here:


### Exercise 2: Engineer and encode

Create a binned feature `bill_band` from `tips['total_bill']` (bins of your choice with `pd.cut`), then one-hot encode the `day` column with `pd.get_dummies`.

In [ ]:
# Your turn. Write your solution here:


### Exercise 3: Rebalance an imbalanced target

The tips `time` column (Lunch/Dinner) is imbalanced. Using `total_bill`, `tip`, `size` as features and `time` as the target, split into train/test (stratified) and oversample the **training** set with `RandomOverSampler`. Print the class balance before and after.

In [ ]:
# Your turn. Write your solution here:


<a id="challenge"></a>
## Challenge (optional): a complete, leakage-safe pipeline on tips

Build a `ColumnTransformer` that standardizes the numeric `tips` features (`total_bill`, `tip`, `size`) and one-hot encodes the categorical ones (`sex`, `smoker`, `day`), to predict `time`. Split (stratified), fit on the **training** set only, transform both, and print the final shapes.

In [ ]:
# Your turn. Write your solution here:


<a id="takeaways"></a>
## Key Takeaways

| Concept / command | What it does |
|---|---|
| `SimpleImputer(strategy='mean')` | Fill missing numeric values (use `median` for skew) |
| `SimpleImputer(strategy='most_frequent')` | Fill missing categorical values with the mode |
| `MinMaxScaler` | Normalize each feature to 0-1 (bounded, outlier-sensitive) |
| `StandardScaler` | Standardize to mean 0, std 1 (the usual default) |
| `pd.cut`, `.dt`, ratios | Feature engineering: bins, date parts, combined features |
| `LabelEncoder` | Integer per category (invents an order; use with care) |
| `pd.get_dummies` / `OneHotEncoder` | 0/1 column per category (safe for nominal data) |
| `RandomOverSampler` / `RandomUnderSampler` | Rebalance an imbalanced target (train only) |
| `train_test_split(..., stratify=y)` | Split before preprocessing; keep class ratios |
| Fit on train, transform on test | The rule that prevents data leakage |
| `Pipeline` + `ColumnTransformer` | Chain preprocessing safely and reproducibly |


## Conclusion

You can now take raw risk data and make it model-ready: impute numeric and categorical gaps, scale, engineer, encode and rebalance, all without leaking information from the test set, and wire it together in a Pipeline. The next notebooks use this prepared data to train and evaluate models.

<a id="reading"></a>
## Further Reading & Resources

- [scikit-learn: Preprocessing data](https://scikit-learn.org/stable/modules/preprocessing.html) scalers, encoders and imputers.
- [scikit-learn: Pipelines and ColumnTransformer](https://scikit-learn.org/stable/modules/compose.html) the leakage-safe way to combine steps.
- [imbalanced-learn](https://imbalanced-learn.org/stable/) over-sampling, under-sampling and SMOTE.